# Phase 6 · Notebook 02 — Ensemble Evaluation

The single biggest reliable lift in NER is usually an ensemble of
independent predictors. The ensemble's job is to absorb the strengths
of each member while averaging out their idiosyncratic mistakes.

We combine:

- **spaCy `en_core_web_trf`** — strong general-purpose English NER (Phase 1 anchor).
- **LegalBERT fine-tuned on TAB train** — produced by Notebook 01.
- **Microsoft Presidio** — NER + the custom CASE_NUMBER regex from Phase 2.

These three are *complementary* by design: spaCy is broad and unbiased,
LegalBERT is legal-domain-specific, Presidio's regex layer catches the
structured identifiers neither NER catches reliably.

Voting policy: keep any span supported by ≥1 predictor (union — recall-
maximising). Drop to `min_votes=2` later if precision becomes a problem.

---


In [ ]:
# ── Run me first if you're on Colab (skip locally — already in requirements.txt) ──
# !pip -q install transformers datasets evaluate seqeval accelerate spacy
# !python -m spacy download en_core_web_lg


## Setup


In [ ]:
import sys
sys.path.insert(0, "../../src")

import time
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from anonymisation.data import load_tab
from anonymisation.mapping import SPACY_TO_TAB
from anonymisation.evaluation import evaluate_document, merge_results, results_to_dataframe
from anonymisation.ensemble import EnsemblePredictor
from anonymisation.predictors import (
    make_hf_predictor, make_presidio_predictor, make_finetuned_predictor,
    build_presidio_analyzer,
)
from anonymisation.device import best_device


## Build the three predictors


In [ ]:
# --- spaCy ---
import spacy
SPACY_MODEL = "en_core_web_trf"
print(f"Loading spaCy: {SPACY_MODEL}")
nlp = spacy.load(SPACY_MODEL)

def spacy_predict(text):
    return [
        (e.start_char, e.end_char, SPACY_TO_TAB[e.label_], e.text)
        for e in nlp(text).ents if e.label_ in SPACY_TO_TAB
    ]


In [ ]:
# --- LegalBERT fine-tuned (from Notebook 01) ---
LEGALBERT_DIR = "checkpoints/legalbert-tab/final"

from transformers import AutoModelForTokenClassification, AutoTokenizer
device, _ = best_device()
print(f"Loading LegalBERT from {LEGALBERT_DIR} on {device}")
lb_tok = AutoTokenizer.from_pretrained(LEGALBERT_DIR)
lb_model = AutoModelForTokenClassification.from_pretrained(LEGALBERT_DIR)
legalbert_predict = make_finetuned_predictor(lb_model, lb_tok, device=device)


In [ ]:
# --- Presidio with the custom CASE_NUMBER recogniser ---
print("Building Presidio analyzer (en_core_web_lg backbone)")
analyzer = build_presidio_analyzer(add_case_number_recognizer=True, spacy_model="en_core_web_lg")
presidio_predict = make_presidio_predictor(analyzer)


## Wire the ensemble


In [ ]:
ensemble = EnsemblePredictor(
    predictors={
        "spacy":     spacy_predict,
        "legalbert": legalbert_predict,
        "presidio":  presidio_predict,
    },
    min_votes=1,         # union — keep every span supported by ≥1 predictor
    tie_break="longest", # on overlapping spans, keep the longest
)

# Smoke test on a synthetic sentence
demo = "Maria Petrova (Application no. 12345/67) sued Acme Holdings Ltd in Sofia District Court."
print("Ensemble output:")
for s, e, t, txt in ensemble(demo):
    print(f"  [{t:8s}] {txt!r}  ({s}:{e})")


## Evaluate on TAB test


In [ ]:
dataset = load_tab()
test_docs = list(dataset["test"])
RESULTS_PATH = "../results/ensemble_results.csv"

all_merged = {}
for mode in ["partial", "exact"]:
    print(f"\n--- {mode} match ---")
    per_doc = []
    start = time.time()
    for i, doc in enumerate(test_docs):
        if (i + 1) % 50 == 0:
            elapsed = time.time() - start
            print(f"  {i + 1}/{len(test_docs)}  ({(i + 1)/elapsed:.1f} docs/s)")
        per_doc.append(evaluate_document(ensemble, doc, mode=mode))
    print(f"  done in {time.time() - start:.1f}s")
    all_merged[mode] = merge_results(per_doc)

results_df = results_to_dataframe(all_merged)
results_df.insert(0, "model", "ensemble_v1")
results_df.to_csv(RESULTS_PATH, index=False)
print(f"\nSaved → {RESULTS_PATH}")


## Per-entity results


In [ ]:
from anonymisation.mapping import TAB_TO_SPACY

for mode in ["partial", "exact"]:
    merged = all_merged[mode]
    print(f"\n── {mode.upper()} MATCH ──")
    rows = []
    for et in list(TAB_TO_SPACY.keys()) + ["_ALL"]:
        r = merged[et]
        rows.append({
            "Entity": et if et != "_ALL" else "▶ OVERALL",
            "TP": r.tp, "FP": r.fp, "FN": r.fn,
            "Precision": f"{r.precision:.1%}",
            "Recall":    f"{r.recall:.1%}",
            "F1":        f"{r.f1:.1%}",
        })
    print(pd.DataFrame(rows).to_string(index=False))


## Try the alternative voting policies

`min_votes=1` is the union (most recall). `min_votes=2` is the
intersection-like middle. `min_votes=3` is strict consensus (highest precision,
lowest recall). Run a quick sample to see the precision/recall trade-off.


In [ ]:
sample = test_docs[:50]
for k in (1, 2, 3):
    ensemble.min_votes = k
    per_doc = [evaluate_document(ensemble, d, mode="partial") for d in sample]
    merged = merge_results(per_doc)["_ALL"]
    print(f"  min_votes={k}:  P={merged.precision:.1%}  R={merged.recall:.1%}  F1={merged.f1:.1%}")
ensemble.min_votes = 1  # reset


## What to look for

- **Overall F1 vs LegalBERT alone (Notebook 01).** The ensemble should win on most labels. If it doesn't, the predictors are making correlated errors — diversify the member set (different transformer backbone, larger spaCy model, etc.) before going deeper on hyperparameters.
- **CODE recall.** With Presidio in the mix this should be high regardless of what LegalBERT and spaCy missed.
- **min_votes precision/recall curve.** The intersection regime (`min_votes=2`) usually trades 5–10 points of recall for 3–5 points of precision. If you're noise-sensitive, that might be the right operating point.

Notebook 03 plots all six models (Phase 1 spaCy, Phase 2 HF + Presidio + RoBERTa, Phase 6 LegalBERT + Ensemble) side by side.
